# 3 — Data Preprocessing & Feature Engineering + Final Validation
## Member 1 — Data Cleaning & Quality Preparation

**Objective:** turn the validated raw bookings into an auditable cleaned-data
handoff using the findings in `02_EDA/EDA.ipynb`. This notebook completes Member 1's
responsibilities: missingness, duplicates, invalid records, types, outlier review,
range validation, and before/after checks.

**Workflow:** EDA finding → quality problem → cleaning decision → justification
→ validation. The raw file remains unchanged. The target is `is_canceled`
(0 = not canceled, 1 = canceled); no target values are imputed or recoded.

This is a **cleaning-stage dataset**, not a final modelling matrix. Member 2 owns
feature engineering, selection, leakage-column removal and encoding; Member 3 owns
splitting, train-only fitted transformations, scaling and final ML validation.

### 1. Reproducible setup

Run all cells from the repository root or this notebook's folder. The shared Python
module implements explicit, testable cleaning rules. No median, mode, quantile cap,
frequency encoding, scaler or other learned transformation is fitted here.

In [1]:
from pathlib import Path
import hashlib
import sys
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/raw/hotel_booking.csv').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.clean_hotel_bookings import (
    RAW_PATH, REPORT_DIR, PROCESSED_PATH, COUNT_COLUMNS, DATE_COLUMNS,
    BINARY_COLUMNS, CATEGORY_COLUMNS, PII_COLUMNS, load_raw,
    remove_invalid_and_duplicates, correct_types_and_missing,
    quality_metrics, outlier_report, validate_cleaned, load_cleaned,
)

pd.set_option('display.max_rows', 100)
raw_sha256 = hashlib.sha256(RAW_PATH.read_bytes()).hexdigest()
raw = load_raw()
print(f'Raw shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns')
print(f'Raw SHA-256: {raw_sha256}')

Raw shape: 119,390 rows × 36 columns
Raw SHA-256: 9dbacc12625aa2a9abf63e123e5275eb674d315afa967610fcbc06b72ef882bd


### 2. Baseline quality profile

EDA §§1.12, 2.9 and 3.9 identify four fields with missing values and flag zero-guest
bookings and ADR anomalies. Report only aggregate statistics; guest contact details
are unnecessary for this analysis. Source-row indices in the removal audit are
zero-based data-row positions (CSV line = index + 2).

In [2]:
before = quality_metrics(raw)
missing_before = raw.isna().sum().rename('missing_count').to_frame()
missing_before['missing_pct'] = missing_before['missing_count'] / len(raw) * 100
display(missing_before.loc[missing_before['missing_count'].gt(0)].round(4))
display(before.loc[~before.index.str.startswith('missing_')].to_frame('before'))

,missing_count,missing_pct
children,4,0.0034
country,488,0.4087
agent,16340,13.6862
company,112593,94.3069


,before
rows,119390.000000
columns,36.000000
exact_duplicate_rows,0.000000
matching_non_pii_rows_review_only,31994.000000
invalid_rows_any_rule,185.000000
zero_night_rows_retained,715.000000
zero_adr_rows_retained,1959.000000
waiting_exceeds_lead_time_review_only,0.000000
not_repeated_with_history_review_only,6237.000000
repeated_without_history_review_only,590.000000


### 3. Decisions and justifications

- **Children (4 missing):** exclude these incomplete guest-composition records
  (approximately 0.0034% of the source). Unknown children are not assumed to be zero.
  This small, target-independent complete-case decision avoids a full-dataset
  imputation statistic. It may introduce selection bias; the class-impact report
  below makes its impact visible. If retaining them becomes necessary, Member 3
  should fit a numerical imputer on training data only.
- **Country:** replace missing values with `Unknown`; do not guess a country or fit
  a global mode. **Agent/company:** use `Not_recorded` and categorical ID strings
  instead of floating-point measurements. Missing IDs may mean no association or
  an unrecorded association; this label does not assert which explanation is true.
- **Duplicates:** remove only complete, exact raw-row duplicates, keeping the first.
  The current source has none. Matching non-PII fields do not establish duplicate
  bookings; there is no reliable unique booking key. Retain these matches and report
  them for Member 3's group/split review.
- **Zero total guests:** exclude records where adults + children + babies = 0
  under the project's guest-booking scope. Retain zero-adult records with children
  or babies. Total guests is calculated for validation only, not added as a feature.
- **Negative ADR:** exclude the one negative-rate record under an explicit
  non-negative accommodation-rate policy. A credit adjustment is possible, but
  cannot be verified here. This is a project policy, not a claim of a confirmed typo.
  Retain zero ADR and positive extremes, including 5400; no justified upper limit
  was established by EDA.
- **Other domain checks:** reject missing, negative, fractional or non-finite counts,
  invalid binary values, invalid arrival calendar dates/week numbers, and malformed
  status dates. Overlapping reasons are audited once per excluded row. Invalid ID
  codes or unexpected remaining missingness stop validation for review.
- **Types:** guest/count/date-component fields become integers, binary fields become
  integer flags, nominal fields become categories, and status date becomes datetime.
  The source status dates use month/day/year; exports use ISO year-month-day.
  Contact/identifier fields stay textual. No predictive date feature is created.
- **Unusual but unverified values:** keep zero-night stays, rare/`Undefined` categories,
  repeat/history disagreements and large valid counts. There is insufficient
  evidence to relabel them or equate `meal=Undefined` with another meal category.

In [3]:
eligible, removal_audit = remove_invalid_and_duplicates(raw)
cleaned = correct_types_and_missing(eligible)
print(f'Removed {len(removal_audit):,} unique rows; retained {len(cleaned):,}.')
display(removal_audit['reason'].value_counts().rename_axis('reason').to_frame('rows'))
display(pd.DataFrame({
    'country_Unknown': [int(cleaned['country'].eq('Unknown').sum())],
    'agent_Not_recorded': [int(cleaned['agent'].eq('Not_recorded').sum())],
    'company_Not_recorded': [int(cleaned['company'].eq('Not_recorded').sum())],
}))

Removed 185 unique rows; retained 119,205.


,rows
reason,
zero_guests,180
missing_children; invalid_counts,4
invalid_adr,1


,country_Unknown,agent_Not_recorded,company_Not_recorded
0,478,16278,112437


### 4. Outliers and range review

EDA §1.11 shows that several count variables have zero IQR, so a Tukey rule flags
every positive count. EDA §3.4 also warns against removing extreme ADR values
automatically. Below, raw-data IQR fences are **descriptive diagnostics only**;
the same fences show before/after counts. They never determine retained rows or
transformed values. Minima/maxima and integer checks verify the cleaned ranges.
Learned clipping, transformations or scaling remain train-only decisions for later
work. Positive extremes remain unresolved observations, not verified errors.

In [4]:
outliers = outlier_report(raw, cleaned)
display(outliers.drop(columns='action').round(3))
schema = pd.DataFrame({
    'column': raw.columns,
    'before_dtype': [str(raw[c].dtype) for c in raw.columns],
    'after_dtype': [str(cleaned[c].dtype) for c in raw.columns],
    'missing_before': [int(raw[c].isna().sum()) for c in raw.columns],
    'missing_after': [int(cleaned[c].isna().sum()) for c in raw.columns],
})
display(schema)

,column,q1_raw,q3_raw,lower_fence_diagnostic_only,upper_fence_diagnostic_only,flagged_before,flagged_after_same_fences,min_before,max_before,min_after,max_after
0,lead_time,18.00,160.0,-195.000,373.000,3005,3004,0.00,737.0,0.0,737.0
1,stays_in_weekend_nights,0.00,2.0,-3.000,5.000,265,258,0.00,19.0,0.0,19.0
2,stays_in_week_nights,1.00,3.0,-2.000,6.000,3354,3330,0.00,50.0,0.0,50.0
3,adults,2.00,2.0,2.000,2.000,29710,29529,0.00,55.0,0.0,55.0
4,children,0.00,0.0,0.000,0.000,8590,8590,0.00,10.0,0.0,10.0
5,babies,0.00,0.0,0.000,0.000,917,917,0.00,10.0,0.0,10.0
6,previous_cancellations,0.00,0.0,0.000,0.000,6484,6479,0.00,26.0,0.0,26.0
7,previous_bookings_not_canceled,0.00,0.0,0.000,0.000,3620,3612,0.00,72.0,0.0,72.0
8,booking_changes,0.00,0.0,0.000,0.000,18076,17977,0.00,21.0,0.0,18.0
9,days_in_waiting_list,0.00,0.0,0.000,0.000,3698,3693,0.00,391.0,0.0,391.0


,column,before_dtype,after_dtype,missing_before,missing_after
0,hotel,str,category,0,0
1,is_canceled,int64,int8,0,0
2,lead_time,int64,int64,0,0
3,arrival_date_year,int64,int64,0,0
4,arrival_date_month,str,category,0,0
5,arrival_date_week_number,int64,int64,0,0
6,arrival_date_day_of_month,int64,int64,0,0
7,stays_in_weekend_nights,int64,int64,0,0
8,stays_in_week_nights,int64,int64,0,0
9,adults,int64,int64,0,0


### 5. Before/after quality and target preservation

Missing-value counts below describe actual nulls. The categorical sentinel labels
retain the fact that the original value was unknown. `review_only` metrics are
observations, not failed validation rules. Matching non-PII rows are retained.
Class proportions can change when invalid rows are excluded; every retained row
must preserve its original target label. No class balancing is performed.

In [5]:
after = quality_metrics(cleaned)
quality = pd.concat([before.rename('before'), after.rename('after')], axis=1)
quality['change'] = quality['after'] - quality['before']
quality.index.name = 'metric'
display(quality.loc[~quality.index.str.startswith('missing_')].round(6))
validation = validate_cleaned(raw, cleaned, removal_audit)
display(validation)

class_impact = pd.DataFrame({
    'before': raw['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
    'after': cleaned['is_canceled'].value_counts().reindex([0, 1], fill_value=0),
})
class_impact['removed'] = class_impact['before'] - class_impact['after']
class_impact['before_pct'] = class_impact['before'] / len(raw) * 100
class_impact['after_pct'] = class_impact['after'] / len(cleaned) * 100
class_impact.index.name = 'is_canceled'
display(class_impact.round(4))

,before,after,change
metric,,,
rows,119390.000000,119205.000000,-185.000000
columns,36.000000,36.000000,0.000000
exact_duplicate_rows,0.000000,0.000000,0.000000
matching_non_pii_rows_review_only,31994.000000,31980.000000,-14.000000
invalid_rows_any_rule,185.000000,0.000000,-185.000000
zero_night_rows_retained,715.000000,645.000000,-70.000000
zero_adr_rows_retained,1959.000000,1810.000000,-149.000000
waiting_exceeds_lead_time_review_only,0.000000,0.000000,0.000000
not_repeated_with_history_review_only,6237.000000,6235.000000,-2.000000


,check,passed
0,Non-empty cleaned dataset,True
1,All original columns preserved for handoff,True
2,No remaining missing values,True
3,No invalid values under documented rules,True
4,No exact duplicate records,True
5,Row counts reconcile with removal audit,True
6,Retained rows and excluded rows partition source,True
7,Target labels unchanged for every retained row,True
8,Counts have integer types,True
9,Binary fields have integer types,True


,before,after,removed,before_pct,after_pct
is_canceled,,,,,
0,75166,75010,156,62.9584,62.9252
1,44224,44195,29,37.0416,37.0748


### 6. Export and validate the saved handoff

`data_quality_report.csv` holds before/after metrics; `preprocessing_summary.csv`
links EDA findings to decisions and validation. Additional reports provide schema,
outlier review, removal reasons and target impact. The cleaned CSV retains all
original columns for the team's next step, including leakage and PII columns.

CSV does not preserve pandas categories or dates. Use `load_cleaned()` to restore
the schema; it loads ID values as strings. Category labels here are storage types,
not a fitted encoder. Later encoders must learn their vocabulary from training data.

In [6]:
decisions = [
    ('children', 'EDA 2.9 / 3.9: four missing counts', 'Exclude incomplete rows',
     'Avoid inventing child counts; tiny target-independent exclusion', 'No missing/fractional/negative counts'),
    ('country', 'EDA 3.5 / 3.9: missing country', 'Fill Unknown',
     'Preserve uncertainty without a fitted global mode', 'No null country values'),
    ('agent, company', 'EDA 3.6 / 3.9: missing identifier-like fields', 'Fill Not_recorded; categorical string IDs',
     'Missing association is ambiguous; IDs are not magnitudes', 'No nulls; nominal dtypes'),
    ('duplicates', 'Dataset understanding: zero full duplicates', 'Drop exact raw duplicates only',
     'Matching booking attributes do not prove the same booking', 'Exact duplicates zero; non-PII matches reported'),
    ('guest composition', 'EDA 2.9: zero-guest observations', 'Exclude confirmed zero-total-guest rows',
     'Project scope requires at least one guest', 'Positive total guests; retain children-only bookings'),
    ('adr', 'EDA 3.4: negative, zero and extreme rates', 'Exclude negative/non-finite rates; retain zero/positive extremes',
     'Explicit non-negative rate policy; no verified upper cutoff', 'Finite non-negative ADR; retained values unchanged'),
    ('counts, binaries, dates', 'Dataset dictionary and EDA unusual-value checks', 'Apply domain checks and explicit types',
     'Counts are integers, flags are binary, dates must parse', 'Domain/type validation passes'),
    ('outliers', 'EDA 1.11 / 3.4: zero IQR and skewed distributions', 'Report only; no automatic clipping',
     'Rare observations are not necessarily errors', 'Before/after ranges; retained numeric values unchanged'),
    ('zero nights, Undefined, repeat/history', 'EDA 1.12 / 2.9 / 3.12: unusual observations', 'Retain for documented review',
     'Insufficient evidence to delete or recode', 'Review counts and category preservation'),
    ('is_canceled', 'Dataset target definition', 'Preserve retained-row labels',
     'Cleaning must not change booking outcomes', 'Exact target alignment and class-impact report'),
    ('team handoff', 'EDA 3.12: leakage and prediction-time concerns', 'Defer feature decisions and learned transforms',
     'Member 2 owns features; Member 3 owns split/pipeline', 'Cleaning validation only; not ML-ready'),
]
summary = pd.DataFrame(decisions, columns=[
    'scope', 'eda_finding', 'cleaning_decision', 'justification', 'validation_or_handoff'
])
REPORT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
cleaned.to_csv(PROCESSED_PATH, index=False, date_format='%Y-%m-%d')
reloaded = load_cleaned()
pd.testing.assert_frame_equal(cleaned.reset_index(drop=True), reloaded)
if hashlib.sha256(RAW_PATH.read_bytes()).hexdigest() != raw_sha256:
    raise ValueError('The raw dataset changed during preprocessing.')
validation = pd.concat([validation, pd.DataFrame([
    {'check': 'Saved CSV restores identical values and schema', 'passed': True},
    {'check': 'Raw file SHA-256 unchanged', 'passed': True},
])], ignore_index=True)
quality.to_csv(REPORT_DIR / 'data_quality_report.csv')
summary.to_csv(REPORT_DIR / 'preprocessing_summary.csv', index=False)
schema.to_csv(REPORT_DIR / 'schema_report.csv', index=False)
outliers.to_csv(REPORT_DIR / 'outlier_review.csv', index=False)
removal_audit.to_csv(REPORT_DIR / 'removed_rows.csv', index=False)
validation.to_csv(REPORT_DIR / 'cleaning_validation.csv', index=False)
class_impact.to_csv(REPORT_DIR / 'target_distribution_comparison.csv')
print(f'Saved {len(cleaned):,} rows × {len(cleaned.columns)} columns to {PROCESSED_PATH.relative_to(ROOT)}')
print(f'All {len(validation)} cleaning/export checks passed.')
display(summary)

Saved 119,205 rows × 36 columns to data\processed\hotel_booking_cleaned.csv
All 15 cleaning/export checks passed.


,scope,eda_finding,cleaning_decision,justification,validation_or_handoff
0,children,EDA 2.9 / 3.9: four missing counts,Exclude incomplete rows,Avoid inventing child counts; tiny target-inde...,No missing/fractional/negative counts
1,country,EDA 3.5 / 3.9: missing country,Fill Unknown,Preserve uncertainty without a fitted global mode,No null country values
2,"agent, company",EDA 3.6 / 3.9: missing identifier-like fields,Fill Not_recorded; categorical string IDs,Missing association is ambiguous; IDs are not ...,No nulls; nominal dtypes
3,duplicates,Dataset understanding: zero full duplicates,Drop exact raw duplicates only,Matching booking attributes do not prove the s...,Exact duplicates zero; non-PII matches reported
4,guest composition,EDA 2.9: zero-guest observations,Exclude confirmed zero-total-guest rows,Project scope requires at least one guest,Positive total guests; retain children-only bo...
5,adr,"EDA 3.4: negative, zero and extreme rates",Exclude negative/non-finite rates; retain zero...,Explicit non-negative rate policy; no verified...,Finite non-negative ADR; retained values uncha...
6,"counts, binaries, dates",Dataset dictionary and EDA unusual-value checks,Apply domain checks and explicit types,"Counts are integers, flags are binary, dates m...",Domain/type validation passes
7,outliers,EDA 1.11 / 3.4: zero IQR and skewed distributions,Report only; no automatic clipping,Rare observations are not necessarily errors,Before/after ranges; retained numeric values u...
8,"zero nights, Undefined, repeat/history",EDA 1.12 / 2.9 / 3.12: unusual observations,Retain for documented review,Insufficient evidence to delete or recode,Review counts and category preservation
9,is_canceled,Dataset target definition,Preserve retained-row labels,Cleaning must not change booking outcomes,Exact target alignment and class-impact report


### 7. Member 1 conclusion and next-member handoff

The current source contains **119,390 rows and 36 columns**. The documented rules
exclude **185 rows** (4 missing child counts, 180 zero-guest rows, 1 negative ADR),
leaving **119,205 rows and 36 columns**. There are no remaining nulls, exact duplicate
records or violations of the declared cleaning rules. Valid extremes and ambiguous
booking-history records remain unchanged. The executed reports above are the
authoritative results if the source changes.

**Member 2:** remove `reservation_status` and `reservation_status_date` from predictors;
exclude `name`, `email`, `phone-number`, and `credit_card`; review prediction-time
availability of `assigned_room_type`, `booking_changes`, waiting duration and requests.
Investigate/define `total_stay`, `total_guests`, feature selection and categorical
encoding. No new predictor features have been added by Member 1.

**Member 3:** establish the prediction point and suitable split. Review identical
booking profiles and customer/group dependence before splitting, since exact-row
cleaning alone cannot guarantee independent train/test sets. Fit imputation (if
needed later), rare-category grouping, encoding, scaling and any outlier treatment
on training data only. Validate final X/y alignment, train/test shapes and leakage
removal. Do not use the entire cleaned CSV directly as X.

Final modelling readiness and model accuracy are outside this Member 1 deliverable.